In [116]:
import pandas as pd
import numpy as np

In [117]:
emails = pd.read_csv('C:\\Users\\MadanRajMA\\Documents\\DS-Proj\\ds-mini-project\\data\\raw\\emails.csv', low_memory=False)

emails.shape

(123389, 27)

Standardizing Column Names

In [118]:
emails.columns = (emails.columns.str.lower())
emails.columns

Index(['co_ref', 'time_to_renewal', 'crm_accreditation_completed',
       'crm_timely_completion', 'crm_progress_towards_accreditation',
       'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
       'crm_contractor_engagement', 'crm_contractor_sentiment',
       'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned',
       'crm_customer_payment_intention', 'crm_competitors_mentioned',
       'crm_membership_level', 'crm_platform_issues_raised',
       'crm_agent_chased_contractor', 'crm_agent_chase_count',
       'crm_accreditation_issues', 'crm_membership_overdue',
       'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
       'crm_customer_complained', 'crm_refund_mentioned',
       'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
       'crm_financial_hardship_mentioned', 'year'],
      dtype='object')

Filtering Churn Cases (14_out)

In [119]:
emails['time_to_renewal'].value_counts()

time_to_renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

In [120]:
emails= emails[emails["time_to_renewal"] == "14_out"]

Checking duplicates

In [121]:
emails.duplicated().sum()

np.int64(0)

In [122]:
emails['co_ref'].isnull().sum()

np.int64(0)

In [123]:
emails['time_to_renewal'].value_counts()

time_to_renewal
14_out    32493
Name: count, dtype: int64

Dropping irrelevant columns<br>
year -> can be derived

In [124]:
emails['year'].unique()

array([2025, 2026])

In [125]:
emails = emails.drop('year', axis=1)
emails= emails.drop('time_to_renewal', axis=1)
emails.shape

(32493, 25)

Handling missing values 

In [126]:
emails= emails.apply(lambda col: col.fillna("Unknown"))

Standardized Categorical values

In [127]:
emails['crm_delays_in_accreditation'].unique()

array(['No', 'Yes', 'Not Discussed',
       "Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)",
       'Not Discussed is not applicable here as there is no mention of accreditation delays, however, the answer to whether there are any delays based on the content provided is: No',
       'Unknown',
       'Not Discussed (However, considering the context is about a policy and a phone number with no answer, it might imply a delay or issue, but strictly speaking, delays in accreditation are not directly mentioned.)',
       "Not Discussed, however, I can infer that there might be a delay since it's been 27+ days with no call, so I will answer: Yes",
       '"Not Discussed (However, since there is no information about accreditation delays, the most fitting answer from the options provided is ""No"")"',
       'Not Discussed is not applicable here as there is no information about delays, so the answer should be: No',
       '"Not Dis

In [128]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Not Discussed (However, I must answer Yes or No, so I'll choose No as there's no clear indication of delays)": "No",

    "Not Discussed is not applicable here as there is no mention of accreditation delays, however, the answer to whether there are any delays based on the content provided is: No": "No",

    "Not Discussed (However, considering the context is about a policy and a phone number with no answer, it might imply a delay or issue, but strictly speaking, delays in accreditation are not directly mentioned.)": "Not Discussed",

    "Not Discussed, however, I can infer that there might be a delay since it's been 27+ days with no call, so I will answer: Yes": "Yes",

    '"Not Discussed (However, since there is no information about accreditation delays, the most fitting answer from the options provided is ""No"")"': "No",

    "Not Discussed is not applicable here as there is no information about delays, so the answer should be: No": "No",

    '"Not Discussed (However, since the question requires a Yes/No answer, I\'ll provide ""No"" as there\'s no mention of delays)"': "No",

    "Not Discussed is not an option for this prompt, I will choose No as there is no mention of delays": "No",

    "Not Discussed is not an option here so I will default to No": "No",

    "Not Discussed but potentially yes as a reason for the AR call": "Yes",

    "Not Discussed is not an option here so the most suitable answer is No": "No"
}
emails["crm_delays_in_accreditation"] = emails["crm_delays_in_accreditation"].replace(mapping)

In [129]:
emails['crm_contractor_engagement'].unique()

array(['No', 'Yes', 'Unknown',
       ' indicating dissatisfaction with the service or support provided."',
       ' which they claim to have requested last year."',
       '"" indicating dissatisfaction with the service."',
       '"" making it not feasible to renew."',
       '"" implying they do not wish to renew their membership."',
       ' indicating uncertainty about continuing with the service."',
       ' and they have chosen to drop clients that require the certification."',
       ' irrelevant requests', ' citing it\'s not the right time."',
       ' implying they want to leave due to an automatic renewal issue."',
       'Not Discussed'], dtype=object)

In [130]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    ' indicating dissatisfaction with the service or support provided."': "No",
    '"" indicating dissatisfaction with the service."': "No",
    '"" making it not feasible to renew."': "No",
    '"" implying they do not wish to renew their membership."': "No",
    ' indicating uncertainty about continuing with the service."': "No",
    ' and they have chosen to drop clients that require the certification."': "No",
    ' citing it\'s not the right time."': "No",
    ' implying they want to leave due to an automatic renewal issue."': "No",

    ' which they claim to have requested last year."': "Unknown",
    ' irrelevant requests': "Unknown"
}
emails["crm_contractor_engagement"] = (emails["crm_contractor_engagement"].map(mapping).fillna("Unknown"))

In [131]:
emails['crm_contractor_sentiment'].unique()

array(['Not Discussed', 'Neutral', 'Satisfied', 'Dissatisfied', 'Unknown',
       'Yes', 'Initially Dissatisfied, later Neutral',
       ' and no work gained from the membership."', 'No'], dtype=object)

In [132]:
mapping = {
    "Satisfied": "Satisfied",
    "Neutral": "Neutral",
    "Dissatisfied": "Dissatisfied",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",

    "Initially Dissatisfied, later Neutral": "Neutral",

    "Yes": "Dissatisfied",
    "No": "Dissatisfied",
    " and no work gained from the membership.\"": "Dissatisfied"
}

emails["crm_contractor_sentiment"] = (
    emails["crm_contractor_sentiment"]
    .map(mapping)
    .fillna("Unknown")
)

In [133]:
emails['crm_contractor_sentiment_score'].unique()


array(['Not Discussed', '50', '80', '30', '40', '20', '100', '0', '90',
       '39', '82', '99', '97', '43', '48', '83', 'Unknown', '70', '27',
       '37', 'Dissatisfied', 'Neutral', 'Yes', '57.5', '62.5', '27.5'],
      dtype=object)

In [134]:
import pandas as pd
import numpy as np

# Step 1: convert to numeric (invalid → NaN)
emails["crm_contractor_sentiment_score"] = pd.to_numeric(
    emails["crm_contractor_sentiment_score"], errors="coerce"
)

# Step 2: convert to int (use -1 for missing)
emails["crm_contractor_sentiment_score"] = (
    emails["crm_contractor_sentiment_score"]
    .fillna(-1)
    .astype(int)
)

# Step 3: convert score → category
def score_to_category(score):
    if score == -1:
        return "Unknown"
    elif score <= 40:
        return "Dissatisfied"
    elif score <= 60:
        return "Neutral"
    elif score <= 100:
        return "Satisfied"
    else:
        return "Unknown"

emails["sentiment_category"] = emails["crm_contractor_sentiment_score"].apply(score_to_category)

#Convert Numeric Score to Categorical
emails["crm_contractor_sentiment_score"] = emails["crm_contractor_sentiment_score"].apply(
    lambda x: "Unknown" if x == -1
    else "Dissatisfied" if x <= 40
    else "Neutral" if x <= 60
    else "Satisfied"
)

# Step 4: preserve "Not Discussed" from original column
emails.loc[
    emails["crm_contractor_sentiment"] == "Not Discussed",
    "sentiment_category"
] = "Not Discussed"

In [135]:
emails['crm_dts_or_ssip_mentioned'].unique()

array(['No', 'Yes', 'Unknown', '20', '0', '50', 'Dissatisfied',
       'Not Discussed'], dtype=object)

In [136]:
emails["crm_dts_or_ssip_mentioned"] = emails["crm_dts_or_ssip_mentioned"].apply(
    lambda x: "Unknown" if pd.isna(x)
    else "Yes" if str(x).strip().lower() == "yes"
    else "Not Discussed" if str(x).strip().lower() == "not discussed"
    else "Unknown" if str(x).strip().lower() == "unknown"
    else "No"
)

In [137]:
emails['crm_customer_payment_intention'].unique()

array(['Not Discussed', 'Yes', 'No', 'Unknown', '0'], dtype=object)

In [138]:
emails["crm_customer_payment_intention"] = emails["crm_customer_payment_intention"].replace({"0": "No"})

In [139]:
emails['crm_agent_chase_count'].unique()

array(['1', '0', '2', '4', '3', '19', '5', 'A few',
       'Quite a few times (no exact number provided)', 'Every month',
       'Not explicitly stated, but implied to be multiple times',
       'Not specified', 'Unknown', 'Not Discussed',
       'Not applicable (only one attempt to contact)',
       'Not applicable (only one email)', 'Multiple', 'Several', '7',
       'Multiple times (implied, but exact number not specified)',
       'Multiple times', 'Not applicable (only one chase)',
       'Not applicable (multiple agents involved)',
       'Multiple times (not specified exactly)', 'A few times', 'Monthly',
       'Numerous occasions (no specific number given)',
       '"Not specified, but mentioned ""a number of our technical support officers"""',
       '"Not specified, but mentioned ""throughout the year"""', '8+',
       '"Not specified, but mentioned ""he didn\'t return my calls or emails"" implying multiple attempts."',
       'Not applicable (only one message)', 'A few tries

In [140]:
emails["crm_agent_chase_count"].value_counts()

crm_agent_chase_count
0                                                                                                     14643
1                                                                                                     10607
2                                                                                                      3269
Unknown                                                                                                3219
3                                                                                                       575
4                                                                                                        77
Not Discussed                                                                                            27
5                                                                                                        17
Several                                                                                                   8
8     

In [141]:
def map_chase_count(val):
    if pd.isna(val):
        return "Unknown"
    
    val = str(val).lower()

    # numeric
    try:
        num = int(val)
        if num <= 1:
            return "Low"
        elif num <= 4:
            return "Medium"
        else:
            return "High"
    except:
        pass

    # text mapping
    if "one" in val:
        return "Low"
    
    elif any(word in val for word in ["few", "several", "monthly", "every month"]):
        return "Medium"
    
    elif any(word in val for word in ["multiple", "many", "numerous", "more than", "8+", "quite a few"]):
        return "High"
    
    elif any(word in val for word in ["unknown", "not specified", "not discussed"]):
        return "Unknown"
    
    return "Unknown"


emails["crm_agent_chase_count"] = emails["crm_agent_chase_count"].apply(map_chase_count)

In [142]:
emails['crm_membership_overdue'].unique()

array(['Not Discussed', 'No', 'Yes',
       ' preventing them from renewing their membership."',
       ' possibly because it was too early."',
       ' with all questions in part one greyed out and part two showing as complete."',
       'Unknown', ' HS Direct."',
       ' especially after a fatality incident where the membership meant nothing to the Health and Safety executive."',
       ' which was later clarified as a typo',
       '"" and requested 1-1 support."',
       ' and ""C02-02 Confined Spaces Pre-Entry Checklist.doc"" which was a checklist and not accepted."',
       ' and the agent had to submit it on their behalf."',
       '"" and needed assistance from the agent."'], dtype=object)

In [143]:
emails["crm_membership_overdue"].value_counts()

crm_membership_overdue
Not Discussed                                                                                                    16326
No                                                                                                                8469
Yes                                                                                                               4469
Unknown                                                                                                           3219
 preventing them from renewing their membership."                                                                    1
 possibly because it was too early."                                                                                 1
 with all questions in part one greyed out and part two showing as complete."                                        1
 HS Direct."                                                                                                         1
 especially after a fatal

In [144]:
#values not matching predefined categories were treated as missing and imputed as “Unknown” to handle non-informative text
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}
emails["crm_membership_overdue"] = (
    emails["crm_membership_overdue"]
    .map(mapping)
    .fillna("Unknown")
)

In [145]:
to_drop = [ ' preventing them from renewing their membership."',
                                                                          ' possibly because it was too early."',
                                 ' with all questions in part one greyed out and part two showing as complete."',
                                                                                                  ' HS Direct."',
 ' especially after a fatality incident where the membership meant nothing to the Health and Safety executive."',
                                                                          ' which was later clarified as a typo',
                                                                                '"" and requested 1-1 support."',
              ' and ""C02-02 Confined Spaces Pre-Entry Checklist.doc"" which was a checklist and not accepted."',
                                                             ' and the agent had to submit it on their behalf."',
                                                                     '"" and needed assistance from the agent."']
emails = emails[~emails['crm_membership_overdue'].isin(to_drop)]

In [146]:
emails['crm_auto_renewal_status'].unique()

array(['0', '2', '1', 'Not Discussed', 'Yes', 'Unknown', 'No',
       ' and also had to provide additional information for the accreditation process."'],
      dtype=object)

In [147]:
emails["crm_auto_renewal_status"].value_counts()

crm_auto_renewal_status
0                                                                                  25915
Unknown                                                                             3219
2                                                                                   2164
1                                                                                   1185
Not Discussed                                                                          4
No                                                                                     3
Yes                                                                                    2
 and also had to provide additional information for the accreditation process."        1
Name: count, dtype: int64

In [148]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "2": "Unknown",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_auto_renewal_status"] = (
    emails["crm_auto_renewal_status"]
    .map(mapping)
    .fillna("Unknown")
)

In [149]:
to_drop = ['and also had to provide additional information for the accreditation process."']
emails = emails[~emails['crm_auto_renewal_status'].isin(to_drop)]

In [150]:
emails['crm_dissatisified_with_renewal_price'].unique()

array(['Not Discussed', 'No', 'Yes', '0', 'Unknown'], dtype=object)

In [151]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "1": "Yes",
    "0": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown"
}

emails["crm_dissatisified_with_renewal_price"] = (
    emails["crm_dissatisified_with_renewal_price"]
    .map(mapping))

In [152]:
emails['crm_customer_complained'].unique()

array(['No', 'Yes', 'Unknown', 'Not Discussed',
       'Not applicable (no email content provided)',
       'Not applicable (there is no content to analyze)',
       'Not applicable (there is no call transcript or email content to analyze)',
       'Not applicable (no conversation provided)',
       'Not applicable (there is no call mentioned in the email content)',
       'Not Applicable',
       'Not applicable (there is no call transcript provided)'],
      dtype=object)

In [153]:
mapping = {
    "Yes": "Yes",
    "No": "No",
    "Not Discussed": "Not Discussed",
    "Unknown": "Unknown",
    "Not applicable (no email content provided)": "Not Applicable",
    "Not applicable (there is no content to analyze)": "Not Applicable",
    "Not applicable (there is no call transcript or email content to analyze)": "Not Applicable",
    "Not applicable (no conversation provided)": "Not Applicable",
    "Not applicable (there is no call mentioned in the email content)": "Not Applicable",
    "Not Applicable": "Not Applicable",
    "Not applicable (there is no call transcript provided)": "Not Applicable"
}

emails["crm_customer_complained"] = emails["crm_customer_complained"].map(mapping)

In [154]:
emails['crm_refund_mentioned'].unique()

array(['Yes', 'No', 'Unknown',
       '"" with all questions on part one greyed out and part two already showing as complete."',
       ' but it was unclear if it was directed at the agent or not."',
       ' considering it ""highway robbery""."',
       ' and saw no value in membership since Homebase was no longer trading."',
       ' making the renewal process pointless for their organization."',
       'Not Discussed', 'Not applicable (no email content provided)',
       'Not applicable (no conversation provided)'], dtype=object)

In [155]:
def clean_refund_mentions(val):
    val_str = str(val).strip()
    
    #Check for "Not applicable" variations first
    if "not applicable" in val_str.lower():
        return "Not Applicable"
    map_to_yes=['but it was unclear if it was directed at the agent or not."',
                'considering it ""highway robbery""."',
                'and saw no value in membership since Homebase was no longer trading."',
                'making the renewal process pointless for their organization."']
    if any(variation in val_str for variation in map_to_yes):
        return "Yes"
    return val_str
emails['crm_refund_mentioned'] = emails['crm_refund_mentioned'].apply(clean_refund_mentions)

In [156]:
to_drop = ['""with all questions on part one greyed out and part two already showing as complete."']
emails = emails[~emails['crm_refund_mentioned'].isin(to_drop)]


In [157]:
emails['crm_negative_customer_experience'].unique()

array(['Yes', 'Unknown', 'No', 'Not Discussed',
       'Not applicable (no email content provided)'], dtype=object)

In [158]:
to_drop = ['Not applicable (no email content provided)']
emails = emails[~emails['crm_negative_customer_experience'].isin(to_drop)]

In [159]:
emails['crm_dissatisfaction_with_support'].unique()

array(['No', 'Not Discussed', 'Yes',
       ' indicating a negative experience with the process."',
       ' indicating a negative experience."',
       ' indicating a negative experience with the accreditation process."',
       '"" which hindered their ability to update documents."', ' Nick."',
       'Unknown',
       'The contractor experienced difficulties accessing the QA system, which hindered their progress.',
       'The Contractor had a negative experience during the initial call, which led to a awkward interaction.',
       'The Contractor had a negative experience due to the unexpected and substantial price increase, which they felt was unfair and unsustainable for their business.',
       'The contractor had a negative experience due to the unexpected cost increase and feeling misled about the price stability.',
       ' ill do it soon',
       'The Contractor had a negative experience due to unfulfilled promises and inability to achieve accreditation, rendering the proces

In [160]:
def clean_refund_mentions(val):
    val_str = str(val).strip()
    
    #Check for variations 
    variations=["negative experience","negative customer experience"] 
    if any(variation in val_str.lower() for variation in variations):
        return "Yes"
    
    return val_str
emails['crm_dissatisfaction_with_support'] = emails['crm_dissatisfaction_with_support'].apply(clean_refund_mentions)

In [161]:
to_drop = ['Nick."','"" which hindered their ability to update documents."']
emails = emails[~emails['crm_dissatisfaction_with_support'].isin(to_drop)]

In [162]:
emails['crm_financial_hardship_mentioned'].unique()

array(['Not Discussed', 'No', 'Yes', 'Unknown', ' bye""."', 'Yes '],
      dtype=object)

In [163]:
# Define the list of values you want to delete
to_drop = [' bye""."', 'Not applicable (no email content provided)']

# Keep only the rows where the value is NOT in that list
emails = emails[~emails['crm_financial_hardship_mentioned'].isin(to_drop)]


In [164]:
emails['crm_membership_level'].value_counts()

crm_membership_level
Not Discussed                          13798
In progress                             8372
Accredited                              6875
Unknown                                 3219
Members only                             187
Not Accredited                            15
Standard                                  10
Express                                    2
Standard package                           1
Standard-Safe contractor Membership        1
Standard Membership                        1
Standard membership level                  1
Premier                                    1
Band C1 (5-9 employees)                    1
Premier Band B                             1
Band B                                     1
Assisted Band C1                           1
Group Membership                           1
Accredited (Silver level)                  1
Name: count, dtype: int64

In [165]:
#mapping for "Standard" and "Accredited" groups
membership_mapping = {
    "Standard membership level": "Standard",
    "Standard Membership": "Standard",
    "Standard package": "Standard",
    "Accredited (Silver level)": "Accredited"
}
emails['crm_membership_level'] = emails['crm_membership_level'].replace(membership_mapping)
print(emails['crm_membership_level'].value_counts())

crm_membership_level
Not Discussed                          13798
In progress                             8372
Accredited                              6876
Unknown                                 3219
Members only                             187
Not Accredited                            15
Standard                                  13
Express                                    2
Group Membership                           1
Assisted Band C1                           1
Band B                                     1
Band C1 (5-9 employees)                    1
Premier Band B                             1
Premier                                    1
Standard-Safe contractor Membership        1
Name: count, dtype: int64


In [166]:
emails.isnull().sum()[emails.isnull().sum() > 0]

Series([], dtype: int64)

Saving cleaned dataset

In [167]:
import os
os.makedirs('../../data/cleaned', exist_ok=True)
emails.to_csv('../../data/cleaned/emails_cleaned.csv', index=False)
print('Saved! Shape:', emails.shape)


Saved! Shape: (32489, 26)


In [170]:
# Assuming 'df_emails' is your email dataset
# Standardize strings to remove whitespace
text_cols = emails.select_dtypes(include=['object']).columns
emails[text_cols] = emails[text_cols].apply(lambda x: x.str.strip())

In [172]:
# --- A. Binary Mapping (Yes/No/Unknown) ---
# We treat 'Unknown' and 'Not Discussed' as 0 (not occurred)
def map_binary(val):
    if pd.isna(val): return 0
    s = str(val).lower()
    if 'yes' in s: return 1
    return 0

binary_cols = [
    'crm_accreditation_completed', 'crm_timely_completion', 'crm_progress_towards_accreditation',
    'crm_delays_in_accreditation', 'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention', 'crm_competitors_mentioned',
    'crm_platform_issues_raised', 'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned', 'crm_negative_customer_experience',
    'crm_dissatisfaction_with_support', 'crm_financial_hardship_mentioned'
]

for col in binary_cols:
    emails[f'{col}_val'] = emails[col].apply(map_binary)

In [173]:
# --- B. Ordinal Mapping (Sentiment & Chase Count) ---
sentiment_map = {
    'Dissatisfied': 1, 'Unknown': 2, 'Not Discussed': 2, 'Neutral': 2, 'Satisfied': 3
}
emails['sentiment_score'] = emails['crm_contractor_sentiment'].map(sentiment_map).fillna(2)

chase_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Unknown': 0, 'Not Discussed': 0}
emails['chase_score'] = emails['crm_agent_chase_count'].map(chase_map).fillna(0)

In [174]:
# Create the aggregated customer profile
email_agg = emails.groupby('co_ref').agg(
    total_emails         = ('co_ref', 'count'),
    
    # Sentiment: What was the worst they felt and how did they feel in the last email?
    avg_sentiment        = ('sentiment_score', 'mean'),
    final_sentiment      = ('sentiment_score', 'last'),
    min_sentiment        = ('sentiment_score', 'min'), # Identifies if they were EVER dissatisfied
    
    # Chasing: Average intensity of agent follow-ups
    avg_chase_intensity  = ('chase_score', 'mean'),
    
    # Issue Flags: How many times were these specific issues raised?
    complaint_count      = ('crm_customer_complained_val', 'sum'),
    refund_requests      = ('crm_refund_mentioned_val', 'sum'),
    price_dissatisfaction = ('crm_dissatisified_with_renewal_price_val', 'sum'),
    competitor_mentions  = ('crm_competitors_mentioned_val', 'sum'),
    hardship_mentions    = ('crm_financial_hardship_mentioned_val', 'sum'),
    
    # Final Status Flags (Did they end the email chain with these statuses?)
    is_currently_overdue = ('crm_membership_overdue_val', 'last'),
    last_known_payment_intent = ('crm_customer_payment_intention_val', 'last')
).reset_index()

In [175]:
# 3. Create 'Ever' Flags
# Useful for Churn Prediction: Did this ever happen in ANY email?
for col in binary_cols:
    email_agg[f'ever_{col}'] = emails.groupby('co_ref')[f'{col}_val'].transform('max')

print(f"Aggregation complete. Unique customers: {len(email_agg)}")

Aggregation complete. Unique customers: 29275


In [176]:
email_agg

,co_ref,total_emails,avg_sentiment,final_sentiment,min_sentiment,avg_chase_intensity,complaint_count,refund_requests,price_dissatisfaction,competitor_mentions,...,ever_crm_agent_chased_contractor,ever_crm_accreditation_issues,ever_crm_membership_overdue,ever_crm_auto_renewal_status,ever_crm_dissatisified_with_renewal_price,ever_crm_customer_complained,ever_crm_refund_mentioned,ever_crm_negative_customer_experience,ever_crm_dissatisfaction_with_support,ever_crm_financial_hardship_mentioned
0,AA0641,1,2.0,2,2,1.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA0784,1,2.0,2,2,1.0,0,0,0,0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,AA0794,1,2.0,2,2,1.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA0923,1,2.0,2,2,1.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA0994,1,3.0,3,3,1.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29270,ZZ9319,1,2.0,2,2,1.0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29271,ZZ9443,2,2.0,2,2,0.5,0,0,0,0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
29272,ZZ9587,1,2.0,2,2,2.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29273,ZZ9702,1,2.0,2,2,2.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [177]:
email_agg.to_csv('../../data/cleaned/emails_aggregated.csv', index=False)